# Exercise 2 — RSI

The Relative Strength Index (RSI) measures momentum: how fast prices are moving and in which direction. Values above 70 signal overbought conditions; below 30 signal oversold. RSI = 100 − 100 / (1 + RS), where RS = average gain / average loss over the look-back window.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(series, window=20):
    return series.rolling(window=window).mean()
def ema(series, window=20):
    return series.ewm(span=window, adjust=False).mean()

# ── Exercise: implement rsi ───────────────────────────────────────────────────

def rsi(series, window=14):
    """Relative Strength Index (0–100).

    Steps:
      1. delta = series.diff()                 — daily change
      2. gain  = delta.clip(lower=0)           — keep only positive changes
      3. loss  = -delta.clip(upper=0)          — keep only negative changes (positive)
      4. avg_gain = gain.rolling(window).mean()
      5. avg_loss = loss.rolling(window).mean()
      6. rs     = avg_gain / avg_loss
      7. return 100 - (100 / (1 + rs))

    First `window` values are NaN.
    """
    # TODO: implement the 7 steps above
    return pd.Series([float("nan")] * len(series), index=series.index)


### Checks

In [ ]:
checks = 0

# 1 — rsi returns same-length Series
try:
    close = _synthetic()["Close"]
    r = rsi(close, 14)
    assert isinstance(r, pd.Series) and len(r) == len(close)
    checks += 1; print("✅ 1 rsi returns same-length Series")
except Exception as e:
    print("❌ 1:", e)

# 2 — first `window` values are NaN
try:
    close = _synthetic()["Close"]
    r = rsi(close, 14)
    assert r.iloc[:14].isna().all(), "first 14 values should be NaN"
    assert not pd.isna(r.iloc[14]), f"index 14 should be first non-NaN, got {r.iloc[14]}"
    checks += 1; print("✅ 2 first window values are NaN, then non-NaN")
except Exception as e:
    print("❌ 2:", e)

# 3 — all non-NaN RSI values are in [0, 100]
try:
    close = _synthetic()["Close"]
    r = rsi(close, 14)
    non_nan = r.dropna()
    assert (non_nan >= 0).all() and (non_nan <= 100).all(), \
        f"RSI out of range: min={non_nan.min():.2f}, max={non_nan.max():.2f}"
    checks += 1; print("✅ 3 all non-NaN RSI values are in [0, 100]")
except Exception as e:
    print("❌ 3:", e)

# 4 — all-rising prices -> RSI approaches 100
try:
    import warnings
    rising = pd.Series([float(i) for i in range(30)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = rsi(rising, 10)
    non_nan = r.dropna()
    assert (non_nan > 90.0).all(), f"expected RSI>90 for all-rising, got {non_nan.tolist()}"
    checks += 1; print("✅ 4 all-rising prices produce RSI > 90")
except Exception as e:
    print("❌ 4:", e)

# 5 — all-falling prices -> RSI approaches 0
try:
    import warnings
    falling = pd.Series([float(30 - i) for i in range(30)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = rsi(falling, 10)
    non_nan = r.dropna()
    assert (non_nan < 10.0).all(), f"expected RSI<10 for all-falling, got {non_nan.tolist()}"
    checks += 1; print("✅ 5 all-falling prices produce RSI < 10")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
